
# Experiment v12b: how much weight should the synthetic data receive?

Adds to v8b a sensitivity analysis requested by an external review. In v6–v11 the receiver's final score was fitted on 200 real patients
(weight 1 each) and 10,000 targeted synthetic patients (total weight 1 each), so the synthetic data carried **50 times** the weight of
the real data. Here the total synthetic weight is set to $r \times n_{\text{local}}$ with $r \in \{0.25, 1, 5, 10, 50\}$ (arms `C_w*`;
`C_w50` is the previous `C_target`), and a weight is also **selected by 5-fold cross-validation on the 200 local patients only**
(arm `C_cv`; the chosen $r$ is stored in column `ct_choice`). The test set is never used for selection.

## Pre-registered hypotheses
- **W1 (cells with a predicted gain, $\Gamma_{upd}(n) \ge 2$):** the gain over updated model sharing increases with $r$; at $r \le 1$ less than half of the $r = 50$ gain remains.
- **W2 (cells without a predicted gain):** the loss relative to updated model sharing shrinks as $r$ decreases ($r = 1$ loses less than $r = 50$).
- **W3 (cross-validated weight):** `C_cv` has a lower mean regret over all cells than any single fixed weight, and keeps at least 80% of the $r = 50$ gain in cells with a predicted gain.

(All v8 hypotheses and outputs are kept.)


---
# Experiment v8b: synthetic data vs direct reweighting — equivalence, finite samples, density-ratio error

This notebook extends v7 (real outcome, short clinical score) with the comparisons requested in the review. All arms are evaluated on the same real test patients with the same protocol.

| Arm | What it tests |
|---|---|
| updated model sharing (closed testing) | practical baseline |
| `IW_donor` | **direct reweighting at the donor**: the receiver sends its density-ratio classifier (trained on its covariates vs the donor's synthetic covariates); the donor refits its simple model on its **real** patients with these weights. Can the gain be obtained without generating data? |
| `IW_donor_ct` | the same, then closed-testing update at the receiver |
| `local_distill` | the donor's teacher is transferred; the receiver distils it on its own covariates. Is there a simpler transfer of knowledge? |
| `C_target` | targeted synthetic data (ours). Does generation add value beyond the alternatives? |
| `IW_donor_exact`, `C_exact_ratio` | the same two methods with the **true** density ratio (known by design) — isolates the effect of density-ratio estimation |

## Pre-registered hypotheses (cells where the theory predicts a gain)
- **H1 (asymptotic equivalence):** with the exact ratio, reweighting and targeted synthetic data both target $h_R^\ast$; their difference is below 25% of $\Gamma$.
- **H2 (finite-sample equivalence, n = 200):** the same holds with estimated ratios.
- **H3 (density-ratio error):** reported, not predicted: the loss from estimating the ratio, for each carrier.
- **H4 (finite-sample prediction):** the excess risk actually reached by closed testing with $n$ = 200 local patients is predicted better by its finite-sample counterpart (the population donor model updated by a closed test on $n$ simulated patients) than by the asymptotic recalibration used in $\Gamma_{\rm upd}$. This addresses the cell where $\Gamma_{\rm upd}$ under-predicted the gain.
- **Clinical:** calibration slope, calibration-in-the-large and net benefit at 10/15/20% risk thresholds.

**Operational differences (not tested, reported):**
- **Reweighting** needs two rounds per receiver: the donor sends covariates, the receiver returns a classifier, the donor refits.
- **Targeted synthetic data** need one transfer: the same pool serves every receiver, and each receiver targets it locally.
- **Privacy:** the disclosure risk of a teacher-labelled synthetic pool, such as membership inference, is outside the scope of this study.

In [ ]:
%pip install -q sdv xgboost statsmodels ucimlrepo

## 1. Configuration

In [ ]:
DATASET = "natality"              # "natality" or "cdc_diabetes"
NAT_URL, NAT_READ, NAT_N = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/DVS/natality/Nat2023us.zip", 0.02, 10**9
CSV_PATH, TARGET_COL, POSITIVE, DROP_COLS = "x.csv", None, None, []

if DATASET == "natality":
    SCORE = ["Age", "BMI", "nulliparous"]
    SHIFTS = {"BMI (in score)": "BMI", "education (omitted)": "education"}
    POP_N = 5000
else:
    SCORE = ["Age", "BMI", "HighBP"]
    SHIFTS = {"BMI (in score)": "BMI", "general health (omitted)": "GenHlth"}
    POP_N = 20000
SKEWS    = [0.5, 0.7, 0.9]
N_LOCAL, N_DONOR, TEST_N, N_SYN, POOL_MULT = 200, 10000, 9500, 10000, 3
REF_SHARE = 0.20                  # random share of the data reserved for the reference model eta_hat
W_CLIP_Q, CART_MIN_LEAF, ALPHA_CT = 0.99, 5, 0.05

QUICK  = True                     # True: 2 repetitions. False: 20 repetitions.
N_REPS = 2 if QUICK else 20
SEED0  = 2026
OUT_CSV = f"results_v12b_{DATASET}_{'quick' if QUICK else 'full'}.csv"
print(f"{DATASET} QUICK={QUICK} reps={N_REPS} score={SCORE} -> {OUT_CSV}")

## 2. Data and reference model

In [ ]:
import warnings, os, time, itertools
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

NAT_FIELDS = {   # 1-based positions from the NCHS 2023 natality user guide -> python slices
    "MAGER": (74, 76), "BMI": (282, 286), "RF_PDIAB": (312, 313), "RF_PHYPE": (314, 315),
    "RF_GHYPE": (315, 316), "RF_EHYPE": (316, 317), "RF_PPTERM": (317, 318), "RF_INFTR": (324, 325),
    "CIG_0": (252, 254), "PRIORLIVE": (170, 172), "PRIORDEAD": (172, 174), "MEDUC": (123, 124),
    "DPLURAL": (453, 454), "PAY_REC": (435, 436)}

def _zip_lines(zpath):
    # The CDC archive uses Deflate64, which Python's zipfile cannot read; stream it through the system unzip tool instead.
    import zipfile, subprocess, shutil
    try:
        with zipfile.ZipFile(zpath) as z:
            name = [n for n in z.namelist() if n.lower().endswith(".txt")][0]
            with z.open(name) as f:
                yield from f
        return
    except NotImplementedError:
        pass
    if shutil.which("unzip"):
        cmd = ["unzip", "-p", zpath]
    elif shutil.which("7z"):
        cmd = ["7z", "e", "-so", zpath]
    else:
        raise RuntimeError("Install unzip first:  !apt-get -qq install unzip")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, bufsize=1 << 20)
    yield from p.stdout
    if p.wait() != 0:
        raise RuntimeError(f"{cmd[0]} failed while reading {zpath}")

def load_natality():
    import urllib.request, zipfile, random
    cache = "natality2023_sample.csv"
    if not os.path.exists(cache):
        zpath = "Nat2023us.zip"
        if not os.path.exists(zpath):
            print("downloading the 2023 natality file (about 230 MB) ..."); urllib.request.urlretrieve(NAT_URL, zpath)
        rnd = random.Random(SEED0); rows = []
        for line in _zip_lines(zpath):
            if rnd.random() >= NAT_READ: continue
            s = line.decode("latin-1")
            rows.append([s[a:b] for a, b in NAT_FIELDS.values()])
        pd.DataFrame(rows, columns=list(NAT_FIELDS)).to_csv(cache, index=False)
        print(f"read {len(rows)} birth records -> {cache}")
    d = pd.read_csv(cache, dtype=str).apply(lambda c: c.str.strip())
    n0 = len(d)
    rf = ["RF_PDIAB", "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_PPTERM", "RF_INFTR"]
    d = d[(d.DPLURAL == "1") & d[rf].isin(["Y", "N"]).all(axis=1)]
    num = {c: pd.to_numeric(d[c], errors="coerce") for c in ["MAGER", "BMI", "CIG_0", "PRIORLIVE", "PRIORDEAD", "MEDUC", "PAY_REC"]}
    ok = (num["BMI"] < 99) & (num["CIG_0"] < 99) & (num["PRIORLIVE"] < 99) & (num["PRIORDEAD"] < 99) \
         & num["MEDUC"].between(1, 8) & (d.RF_PHYPE == "N")      # women with chronic hypertension excluded
    d = d[ok]; num = {k: v[ok] for k, v in num.items()}
    parity = num["PRIORLIVE"] + num["PRIORDEAD"]
    X = pd.DataFrame({
        "Age": num["MAGER"], "BMI": num["BMI"],
        "nulliparous": (parity == 0).astype(int), "prior_births": parity.clip(upper=6),
        "pregestational_diabetes": (d.RF_PDIAB == "Y").astype(int),
        "previous_preterm": (d.RF_PPTERM == "Y").astype(int), "infertility_treatment": (d.RF_INFTR == "Y").astype(int),
        "cigarettes_before": num["CIG_0"].clip(upper=40), "education": num["MEDUC"]})
    y = ((d.RF_GHYPE == "Y") | (d.RF_EHYPE == "Y")).astype(int)
    out = X.assign(y=y.values).sample(min(NAT_N, len(X)), random_state=SEED0).reset_index(drop=True)
    print(f"natality: {n0} records read | {len(X)} singleton births, no chronic hypertension, complete data | {len(out)} kept")
    return out

def load_raw():
    if DATASET == "natality":
        return load_natality()
    if DATASET == "uci":
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=863)
        X = d.data.features.copy()
        y_raw = d.data.targets.iloc[:, 0].astype(str).str.strip().str.lower()
        y = (y_raw == "high risk").astype(int)          # high risk vs low/mid risk
    else:
        df = pd.read_csv(CSV_PATH).drop(columns=DROP_COLS, errors="ignore")
        tcol = TARGET_COL or df.columns[-1]
        pos = POSITIVE if POSITIVE is not None else df[tcol].value_counts().idxmin()
        y = (df[tcol] == pos).astype(int); X = df.drop(columns=[tcol])
        print(f"Outcome column {tcol!r}, positive value {pos!r}")
    X = X.select_dtypes(include="number")
    X = X[X.columns[X.isna().mean() <= 0.30]]
    return X.assign(y=y.values)

def audit(df):
    if DATASET == "natality":   # registry data: identical rows are different women, so no de-duplication
        feats = [c for c in df.columns if c != "y"]
        print(f"features: {feats}\nprevalence of gestational hypertension/eclampsia: {df.y.mean():.3f}")
        return df.reset_index(drop=True)
    n0 = len(df); df = df.dropna(); n1 = len(df)
    df = df.drop_duplicates(); n2 = len(df)
    feats = [c for c in df.columns if c != "y"]
    conflict = df.groupby(feats)["y"].transform("nunique") > 1
    df = df[~conflict].reset_index(drop=True)
    print(f"rows: raw {n0} | complete {n1} | after removing exact duplicates {n2} "
          f"| after removing {int(conflict.sum())} conflicting rows {len(df)}")
    print(f"features: {feats}\nprevalence: {df.y.mean():.3f}")
    assert SHIFT_VAR in feats, f"SHIFT_VAR {SHIFT_VAR!r} is not a column"
    return df



def load_cdc():
    cache = "cdc_diabetes_binary.csv"
    if os.path.exists(cache):
        df = pd.read_csv(cache)
    else:
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=891)
        df = d.data.features.copy(); df["y"] = d.data.targets["Diabetes_binary"].values
        df.to_csv(cache, index=False)
    df = df.astype(float); df["y"] = df["y"].astype(int)
    print(f"CDC Diabetes Health Indicators: {len(df)} adults, prevalence {df.y.mean():.3f}")
    return df.reset_index(drop=True)

DATA = load_cdc() if DATASET == "cdc_diabetes" else audit(load_raw())
FEATURES = [c for c in DATA.columns if c != "y"]


from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

X_ALL = DATA[FEATURES].to_numpy(float)
Y_ALL = DATA.y.to_numpy()
S_IDX = [FEATURES.index(c) for c in SCORE]
expit = lambda u: 1 / (1 + np.exp(-u))

_perm = np.random.default_rng(SEED0).permutation(len(DATA))
REF = _perm[: int(REF_SHARE * len(DATA))]
IN_REF = np.zeros(len(DATA), bool); IN_REF[REF] = True
ETA_HAT = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.9, verbosity=0, n_jobs=-1) \
    .fit(X_ALL[REF], Y_ALL[REF]).predict_proba(X_ALL)[:, 1]

HIGH, RULES = {}, {}
for name, v in SHIFTS.items():
    col = DATA[v].to_numpy(float); med = np.median(col[~IN_REF])
    h = col > med
    if h[~IN_REF].mean() < 0.3: h = col >= med
    HIGH[name] = h; RULES[name] = (FEATURES.index(v), med, bool(h[~IN_REF].mean() != (col[~IN_REF] > med).mean()))
    print(f"shift '{name}': share above the median = {h[~IN_REF].mean():.2f}")
_rest = ~IN_REF
_ri = np.flatnonzero(_rest); _h = min(20000, len(_ri) // 2); _tr, _te = _ri[:_h], _ri[_h:2 * _h]
_lrf = LogisticRegression(C=1.0, max_iter=5000).fit(X_ALL[_tr], Y_ALL[_tr])
_lrs = LogisticRegression(C=1.0, max_iter=5000).fit(X_ALL[_tr][:, S_IDX], Y_ALL[_tr])
print(f"held-out log-loss: full logistic {log_loss(Y_ALL[_te], _lrf.predict_proba(X_ALL[_te])[:, 1]):.4f} | "
      f"short score {log_loss(Y_ALL[_te], _lrs.predict_proba(X_ALL[_te][:, S_IDX])[:, 1]):.4f} | "
      f"reference XGBoost {log_loss(Y_ALL[_te], ETA_HAT[_te]):.4f}")

## 3. Helpers (same as v6)

In [ ]:
import torch, statsmodels.api as sm
from scipy import stats
from sklearn.tree import DecisionTreeRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

COLS = FEATURES + ["y"]

def cart_synth(df, n, seed):
    rng = np.random.default_rng(seed)
    out = pd.DataFrame({COLS[0]: rng.choice(df[COLS[0]].to_numpy(), n, replace=True)})
    for j, c in enumerate(COLS[1:], 1):
        prev = COLS[:j]
        tree = DecisionTreeRegressor(min_samples_leaf=CART_MIN_LEAF, random_state=seed + j).fit(df[prev], df[c])
        lr_, ls_ = tree.apply(df[prev]), tree.apply(out[prev])
        vals = df[c].to_numpy(); new = np.empty(n)
        groups = pd.Series(np.arange(len(df))).groupby(lr_).apply(np.array)
        for leaf in np.unique(ls_):
            pos = np.where(ls_ == leaf)[0]
            new[pos] = vals[rng.choice(groups[leaf], len(pos), replace=True)]
        out[c] = new
    return out

def _tilt_features(X):
    # features for the density-ratio classifier: all covariates plus squares of the non-binary ones
    nb = [j for j in range(X.shape[1]) if len(np.unique(X_ALL[:2000, j])) > 2]
    return np.c_[X, X[:, nb] ** 2]

def target_resample(pool_X, local_X, n, rng):
    # Targeting done by the RECEIVER on the synthetic pool it received (its own covariates never leave the site):
    # a probabilistic classifier separates local covariates from the pool, and its odds estimate the density ratio p_R / q
    F_pool, F_loc = _tilt_features(pool_X), _tilt_features(local_X)
    clf = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced"))])
    clf.fit(np.vstack([F_loc, F_pool]), np.r_[np.ones(len(F_loc)), np.zeros(len(F_pool))])
    p = np.clip(clf.predict_proba(F_pool)[:, 1], 1e-6, 1 - 1e-6)
    w = p / (1 - p); w = np.minimum(w, np.quantile(w, W_CLIP_Q)); w /= w.sum()
    return rng.choice(len(pool_X), size=n, replace=True, p=w), 1 / np.sum(w ** 2) / len(w)

def fit_lr(X_hard=None, y_hard=None, X_soft=None, p_soft=None, C=1.0):
    # logistic regression on hard local labels and/or soft (teacher-probability) labels: each soft row is
    # duplicated with y=1 (weight p) and y=0 (weight 1-p), which is the infinite-relabelling limit of method B
    Xs, ys, ws = [], [], []
    if X_hard is not None:
        Xs.append(X_hard); ys.append(y_hard); ws.append(np.ones(len(y_hard)))
    if X_soft is not None:
        Xs += [X_soft, X_soft]; ys += [np.ones(len(p_soft)), np.zeros(len(p_soft))]; ws += [p_soft, 1 - p_soft]
    m = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=C, max_iter=5000))])
    return m.fit(np.vstack(Xs), np.concatenate(ys), lr__sample_weight=np.concatenate(ws))

def logit(p):
    p = np.clip(p, 1e-9, 1 - 1e-9); return np.log(p / (1 - p))

def _glm(y, X, offset=None, w=None):
    try:
        r = sm.GLM(y, X, family=sm.families.Binomial(), offset=offset, freq_weights=w).fit()
        if not r.converged or not np.all(np.isfinite(r.params)): return None
        return r
    except Exception:
        return None

def recal(lp_tr, y_tr, lp_te):
    r = _glm(y_tr, sm.add_constant(lp_tr, has_constant="add"))
    return expit(lp_te) if r is None else expit(r.params[0] + r.params[1] * lp_te)

def closed_test(lp_tr, y_tr, lp_te):
    # Closed testing (Vergouwe et al. 2017) among original / recalibration-in-the-large / logistic recalibration.
    # Model revision (re-estimating all coefficients) is not offered: with a few hundred patients it overfits badly.
    ll0 = np.sum(y_tr * np.log(expit(lp_tr)) + (1 - y_tr) * np.log(1 - expit(lp_tr)))
    m1 = _glm(y_tr, np.ones((len(y_tr), 1)), offset=lp_tr)
    m2 = _glm(y_tr, sm.add_constant(lp_tr, has_constant="add"))
    ll = lambda m: -np.inf if m is None else m.llf
    sig = lambda a, b, df: 2 * (ll(a) - b) > stats.chi2.ppf(1 - ALPHA_CT, df)
    if m2 is None or not sig(m2, ll0, 2):
        return expit(lp_te), "original"
    if sig(m2, ll(m1), 1):
        return expit(m2.params[0] + m2.params[1] * lp_te), "recalibration"
    return (expit(m1.params[0] + lp_te), "intercept") if m1 is not None else (expit(lp_te), "original")

def kl_excess(eta, p):
    p = np.clip(p, 1e-9, 1 - 1e-9); e = np.clip(eta, 1e-12, 1 - 1e-12)
    return float(np.mean(e * np.log(e / p) + (1 - e) * np.log((1 - e) / (1 - p))))

def teacher_fit(X, y, seed):
    xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9,
                        eval_metric="logloss", random_state=seed, n_jobs=-1, verbosity=0)
    return CalibratedClassifierCV(xgb, method="sigmoid", cv=5).fit(X, y)



# ---------------- v8 additions ----------------
T_NB = [0.10, 0.15, 0.20]          # risk thresholds for net benefit
B_CT = 30                          # simulations of the closed test with n local patients (finite-sample prediction)

def ratio_classifier(pool_X, local_X):
    # receiver-side density-ratio model: local covariates (1) vs the donor's synthetic covariates (0)
    F_pool, F_loc = _tilt_features(pool_X), _tilt_features(local_X)
    clf = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced"))])
    return clf.fit(np.vstack([F_loc, F_pool]), np.r_[np.ones(len(F_loc)), np.zeros(len(F_pool))])

def est_weights(clf, X):
    p = np.clip(clf.predict_proba(_tilt_features(X))[:, 1], 1e-6, 1 - 1e-6)
    w = p / (1 - p); w = np.minimum(w, np.quantile(w, W_CLIP_Q)); return w / w.mean()

def exact_weights(X, sk, rule):
    # the true density ratio p_R / p_donor of the simulated sites (known by design): sk/(1-sk) above the median, (1-sk)/sk below
    j, thr, ge = rule
    h = X[:, j] >= thr if ge else X[:, j] > thr
    return np.where(h, sk / (1 - sk), (1 - sk) / sk)

def resample_w(w, n, rng):
    p = w / w.sum(); return rng.choice(len(w), size=n, replace=True, p=p), 1 / np.sum(p ** 2) / len(p)

def fit_lr_w(X, y, w):
    m = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000))])
    return m.fit(X, y, lr__sample_weight=w)

def net_benefit(y, p, t):
    pos = p >= t
    return float(np.mean(pos & (y == 1)) - np.mean(pos & (y == 0)) * t / (1 - t))

def cal_stats(y, p):
    lp = logit(p)
    s = _glm(y, sm.add_constant(lp, has_constant="add")); c = _glm(y, np.ones((len(y), 1)), offset=lp)
    return (np.nan if s is None else s.params[1]), (np.nan if c is None else c.params[0])

def ct_finite(lp_fn, X_pop, eta_pop, lp_te, e_te, n, B, rng):
    # expected excess risk of "population donor model + closed test on n local patients": the finite-sample
    # counterpart of Gamma_upd (the asymptotic version assumes perfect local updating)
    out = []
    for _ in range(B):
        i = rng.choice(len(X_pop), n, replace=False)
        yb = (rng.random(n) < eta_pop[i]).astype(int)
        p, _ = closed_test(lp_fn(X_pop[i]), yb, lp_te)
        out.append(kl_excess(e_te, p))
    return float(np.mean(out))

def draw(avail, n, share_high, high, rng):
    nh = int(round(share_high * n))
    idx = np.r_[rng.choice(np.flatnonzero(avail & high), nh, replace=False),
                rng.choice(np.flatnonzero(avail & ~high), n - nh, replace=False)]
    avail[idx] = False
    return rng.permutation(idx)                     # shuffle: draws are built high-then-low, and calibration CV folds must not follow that order

def proj_score(X, p):
    # population I-projection of probabilities p onto the short logistic score
    return fit_lr(X_soft=X[:, S_IDX], p_soft=p, C=1e4)

# ---------------- v12: weight of the synthetic rows relative to the real local rows ----------------
from sklearn.model_selection import StratifiedKFold
R_GRID = [0.25, 1, 5, 10, 50]      # total synthetic weight / number of real local rows (50 = v6-v11 setting: 10,000 rows vs 200)

def fit_lr_a(Xh, yh, Xs, ps, r, C=1.0):
    # local rows weight 1; each soft synthetic row duplicated (y=1, weight a*p) and (y=0, weight a*(1-p)), a = r * n_local / n_syn
    a = r * len(yh) / len(ps)
    Xa = np.vstack([Xh, Xs, Xs]); ya = np.r_[yh, np.ones(len(ps)), np.zeros(len(ps))]
    wa = np.r_[np.ones(len(yh)), a * ps, a * (1 - ps)]
    m = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=C, max_iter=5000))])
    return m.fit(Xa, ya, lr__sample_weight=wa)

def cv_weight(Xh, yh, Xs, ps, seed):
    # choose r by 5-fold CV log-loss on the local patients only; the synthetic sample is the same in every fold
    if min(yh.sum(), len(yh) - yh.sum()) < 5: return 50
    cv = StratifiedKFold(5, shuffle=True, random_state=seed); loss = []
    for r in R_GRID:
        L = 0.0
        for tr, va in cv.split(Xh, yh):
            p = np.clip(fit_lr_a(Xh[tr], yh[tr], Xs, ps, r).predict_proba(Xh[va])[:, 1], 1e-9, 1 - 1e-9)
            L -= np.sum(yh[va] * np.log(p) + (1 - yh[va]) * np.log(1 - p))
        loss.append(L)
    return R_GRID[int(np.argmin(loss))]


## 4. Run (resumes from the checkpoint file)

In [ ]:
COLUMNS = ["dataset", "shift", "share", "rep", "arm", "logloss_y", "brier_y", "auc", "ex_kl_ref", "ct_choice", "ess_C", "cal_slope", "citl", "nb10", "nb15", "nb20"]

def done_keys():
    if not os.path.exists(OUT_CSV): return set()
    r = pd.read_csv(OUT_CSV); return set(zip(r["shift"], r["share"], r["rep"]))

def run_cell(shift, sk, rep):
    t0 = time.time()
    seed = SEED0 + 1000 * rep + 100 * list(SHIFTS).index(shift) + 10 * SKEWS.index(sk)
    rng = np.random.default_rng(seed)
    high = HIGH[shift]
    avail = ~IN_REF.copy()
    te, loc, rpop = draw(avail, TEST_N, sk, high, rng), draw(avail, N_LOCAL, sk, high, rng), draw(avail, POP_N, sk, high, rng)
    don, dpop = draw(avail, N_DONOR, 1 - sk, high, rng), draw(avail, POP_N, 1 - sk, high, rng)
    X, y, S = X_ALL, Y_ALL, S_IDX
    res = {}

    res["local_score"] = fit_lr(X[loc][:, S], y[loc]).predict_proba(X[te][:, S])[:, 1]
    star, dstar = proj_score(X[rpop], ETA_HAT[rpop]), proj_score(X[dpop], ETA_HAT[dpop])
    res["oracle_star"] = star.predict_proba(X[te][:, S])[:, 1]
    res["oracle_D"] = dstar.predict_proba(X[te][:, S])[:, 1]
    rc = fit_lr(X_soft=dstar.decision_function(X[rpop][:, S])[:, None], p_soft=ETA_HAT[rpop], C=1e4)
    res["oracle_D_recal"] = rc.predict_proba(dstar.decision_function(X[te][:, S])[:, None])[:, 1]

    hD = fit_lr(X[don][:, S], y[don])                                  # the donor publishes its short score
    lp_loc, lp_te = hD.decision_function(X[loc][:, S]), hD.decision_function(X[te][:, S])
    res["share_score"] = expit(lp_te)
    res["share_score_recal"] = recal(lp_loc, y[loc], lp_te)
    res["share_score_ct"], ct = closed_test(lp_loc, y[loc], lp_te)
    lpf = lambda Z: dstar.decision_function(Z[:, S])
    extra = {"oracle_D_ct_n": ct_finite(lpf, X[rpop], ETA_HAT[rpop], lpf(X[te]), ETA_HAT[te], N_LOCAL, B_CT, rng)}

    t = teacher_fit(X[don], y[don], seed)                               # teacher sees all predictors
    res["teacher_full"] = t.predict_proba(X[te])[:, 1]
    ddf = pd.DataFrame(X[don], columns=FEATURES).assign(y=y[don])
    pool = cart_synth(ddf, POOL_MULT * N_DONOR, seed)[FEATURES].to_numpy(float)
    p_pool = t.predict_proba(pool)[:, 1]
    clf = ratio_classifier(pool, X[loc])                              # targeting uses all predictors held locally
    idxC, ess = resample_w(est_weights(clf, pool), N_SYN, rng)
    idxE, _ = resample_w(exact_weights(pool, sk, RULES[shift]), N_SYN, rng)
    hIW = fit_lr_w(X[don][:, S], y[don], est_weights(clf, X[don]))
    res["IW_donor"] = hIW.predict_proba(X[te][:, S])[:, 1]
    res["IW_donor_ct"], _ = closed_test(hIW.decision_function(X[loc][:, S]), y[loc], hIW.decision_function(X[te][:, S]))
    res["IW_donor_exact"] = fit_lr_w(X[don][:, S], y[don], exact_weights(X[don], sk, RULES[shift])).predict_proba(X[te][:, S])[:, 1]
    res["C_exact_ratio"] = fit_lr(X[loc][:, S], y[loc], pool[idxE][:, S], p_pool[idxE]).predict_proba(X[te][:, S])[:, 1]
    B = np.arange(N_SYN)
    res["B_untargeted"] = fit_lr(X[loc][:, S], y[loc], pool[B][:, S], p_pool[B]).predict_proba(X[te][:, S])[:, 1]
    res["C_target"] = fit_lr(X[loc][:, S], y[loc], pool[idxC][:, S], p_pool[idxC]).predict_proba(X[te][:, S])[:, 1]
    XL, XS, XT = X[loc][:, S], pool[idxC][:, S], X[te][:, S]
    for r in R_GRID[:-1]:
        res[f"C_w{r:g}"] = fit_lr_a(XL, y[loc], XS, p_pool[idxC], r).predict_proba(XT)[:, 1]
    res["C_w50"] = res["C_target"]
    r_cv = cv_weight(XL, y[loc], XS, p_pool[idxC], seed)
    res["C_cv"] = fit_lr_a(XL, y[loc], XS, p_pool[idxC], r_cv).predict_proba(XT)[:, 1]
    res["local_distill"] = fit_lr(X_soft=X[loc][:, S], p_soft=t.predict_proba(X[loc])[:, 1]).predict_proba(X[te][:, S])[:, 1]

    rows = []
    for arm, p in res.items():
        pc = np.clip(p, 1e-9, 1 - 1e-9)
        rows.append(dict(dataset=DATASET, shift=shift, share=sk, rep=rep, arm=arm,
                         logloss_y=log_loss(y[te], pc, labels=[0, 1]), brier_y=float(np.mean((pc - y[te]) ** 2)),
                         auc=roc_auc_score(y[te], pc), ex_kl_ref=kl_excess(ETA_HAT[te], pc),
                         ct_choice=ct if arm == "share_score_ct" else (str(r_cv) if arm == "C_cv" else ""), ess_C=ess if arm == "C_target" else np.nan,
                         **dict(zip(["cal_slope", "citl"], cal_stats(y[te], pc))),
                         **{f"nb{int(t*100)}": net_benefit(y[te], pc, t) for t in T_NB}))
    for arm, v in extra.items():
        rows.append(dict(dataset=DATASET, shift=shift, share=sk, rep=rep, arm=arm, ex_kl_ref=v))
    pd.DataFrame(rows).reindex(columns=COLUMNS).to_csv(OUT_CSV, mode="a", header=not os.path.exists(OUT_CSV), index=False)
    print(f"  {shift:26s} share {sk:.1f} rep {rep:2d}  {time.time() - t0:5.1f}s")

t_start = time.time()
for rep, shift, sk in itertools.product(range(N_REPS), SHIFTS, SKEWS):
    if (shift, sk, rep) not in done_keys(): run_cell(shift, sk, rep)
print(f"Finished in {(time.time() - t_start) / 60:.1f} min -> {OUT_CSV}")

## 5. Results: four-way comparison, hypotheses, clinical usefulness

## Sensitivity to the weight of the synthetic data

In [ ]:

R12 = pd.read_csv(OUT_CSV)
M12 = "logloss_y"; CELL12 = ["shift", "share"]; CT12 = "share_score_ct"; OR_N, OR_STAR, GM = "oracle_D_ct_n", "oracle_star", "ex_kl_ref"
ARMS12 = [f"C_w{r:g}" for r in R_GRID]
def g12(sub, a, b, m=M12):
    A = sub[sub.arm == a].set_index("rep")[m]; B = sub[sub.arm == b].set_index("rep")[m]
    d = (A - B).dropna() * 1000
    h = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
    return d.mean(), d.mean() - h, d.mean() + h
rows, W1, W2, keep = [], [], [], []
for key, sub in R12.groupby(CELL12):
    pr = sub.groupby("arm")[GM].mean() * 1000
    gam_n = pr[OR_N] - pr[OR_STAR]
    g = {a: g12(sub, CT12, a)[0] for a in ARMS12 + ["C_cv"]}
    ch = sub[sub.arm == "C_cv"].ct_choice.astype(float)
    rows.append(dict(zip(CELL12, key), Gamma_upd_n=round(gam_n, 1), **{a: round(v, 1) for a, v in g.items()},
                     cv_minus_w50=f"{g12(sub, 'C_w50', 'C_cv')[0]:+.1f} [{g12(sub, 'C_w50', 'C_cv')[1]:+.1f}, {g12(sub, 'C_w50', 'C_cv')[2]:+.1f}]",
                     chosen_r_median=ch.median(), chosen_r_50_share=round((ch == 50).mean(), 2)))
    if gam_n >= 2:
        W1.append(g["C_w1"] < 0.5 * g["C_w50"] and all(g[ARMS12[i]] <= g[ARMS12[i + 1]] + 1.0 for i in range(len(ARMS12) - 1)))
        keep.append(g["C_cv"] >= 0.8 * g["C_w50"])
    else:
        W2.append(g["C_w1"] > g["C_w50"])
T12 = pd.DataFrame(rows)
print("Gain over updated model sharing by synthetic weight r (x1000, positive = better than updated sharing)")
display(T12)
# regret over the fixed-weight grid, per cell and repetition
piv = R12[R12.arm.isin(ARMS12 + ["C_cv"])].pivot_table(index=CELL12 + ["rep"], columns="arm", values=M12)
best = piv[ARMS12].min(axis=1)
reg = (piv.sub(best, axis=0) * 1000).mean().round(2)
print("\nMean regret relative to the best fixed weight in each cell and repetition (x1000):"); print(reg.to_string())
W3 = reg["C_cv"] < reg[ARMS12].min() and all(keep)
print(f"\nW1 (gain grows with r; r<=1 keeps < half): {all(W1)} ({sum(W1)}/{len(W1)} cells)\n"
      f"W2 (smaller r loses less where no gain): {all(W2)} ({sum(W2)}/{len(W2)} cells)\n"
      f"W3 (CV weight: lowest regret and >= 80% of r=50 gain): {W3}  (regret C_cv {reg['C_cv']} vs best fixed {reg[ARMS12].min()}; keep {sum(keep)}/{len(keep)})")
display(R12[R12.arm.isin([CT12, "C_w1", "C_w50", "C_cv"])].groupby(CELL12 + ["arm"])[["brier_y", "auc", "cal_slope"]].mean().round(3))


In [ ]:
R = pd.read_csv(OUT_CSV)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)
M = "logloss_y"                     # primary metric
CELL = ["shift", "share"]
def gain(sub, a, b, m=M):
    A = sub[sub.arm == a].set_index("rep")[m]; B = sub[sub.arm == b].set_index("rep")[m]
    d = (A - B).dropna() * 1000
    if len(d) < 2: return d.mean(), np.nan, np.nan
    h = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
    return d.mean(), d.mean() - h, d.mean() + h
f = lambda g: f"{g[0]:+.2f} [{g[1]:+.2f}, {g[2]:+.2f}]"
KL = "ex_kl_ref"
rows = []
for key, sub in R.groupby(CELL):
    pr = sub.groupby("arm")[KL].mean() * 1000
    G_upd = pr["oracle_D_recal"] - pr["oracle_star"]; G_n = pr["oracle_D_ct_n"] - pr["oracle_star"]; G = pr["oracle_D"] - pr["oracle_star"]
    rows.append(dict(zip(CELL, key), Gamma=round(G, 2), Gamma_upd=round(G_upd, 2), Gamma_upd_n=round(G_n, 2),
        IW_donor=f(gain(sub, "share_score_ct", "IW_donor")), IW_donor_ct=f(gain(sub, "share_score_ct", "IW_donor_ct")),
        IW_exact=f(gain(sub, "share_score_ct", "IW_donor_exact")), C_target=f(gain(sub, "share_score_ct", "C_target")),
        C_exact=f(gain(sub, "share_score_ct", "C_exact_ratio")), local_distill=f(gain(sub, "share_score_ct", "local_distill")),
        C_minus_IW=f(gain(sub, "IW_donor", "C_target")), Cexact_minus_IWexact=f(gain(sub, "IW_donor_exact", "C_exact_ratio"))))
T = pd.DataFrame(rows)
print("Gains over updated model sharing (closed testing), real-outcome log-loss (nats) x 1000; positive = arm better.")
print("C_minus_IW > 0 means targeted synthetic data beat donor-side reweighting.")
display(T)

# ---- pre-registered hypotheses ----
inter = [r for r in rows if r["Gamma_upd"] >= 2 or r["Gamma_upd_n"] >= 2]
def ci(sub, a, b): return gain(sub, a, b)
H = {}
for r in inter:
    sub = R[np.all([R[c] == r[c] for c in CELL], axis=0)]
    g_pred = max(r["Gamma"], 1e-9)
    d_ex = ci(sub, "IW_donor_exact", "C_exact_ratio"); d_est = ci(sub, "IW_donor", "C_target")
    H[tuple(r[c] for c in CELL)] = dict(
        H1_asymptotic_equiv=bool(abs(d_ex[0]) < 0.25 * g_pred),
        H2_finite_equiv=bool(abs(d_est[0]) < 0.25 * g_pred),
        H3_ratio_cost_C=round(gain(sub, "C_exact_ratio", "C_target")[0], 2),
        H3_ratio_cost_IW=round(gain(sub, "IW_donor_exact", "IW_donor")[0], 2))
print("\nH1: exact-ratio reweighting and exact-ratio synthetic within 25% of Gamma | H2: same with estimated ratios | "
      "H3: cost of estimating the density ratio (negative = estimation hurts)")
display(pd.DataFrame(H).T)
pr_all = R.groupby(CELL + ["arm"])[KL].mean().unstack("arm") * 1000
ct_obs = pr_all["share_score_ct"]; ct_n = pr_all["oracle_D_ct_n"]; ct_inf = pr_all["oracle_D_recal"]
print("\nH4: how well do we predict what closed testing achieves with n local patients? (excess KL x 1000)")
display(pd.DataFrame({"observed closed test": ct_obs.round(2), "finite-sample prediction": ct_n.round(2),
                      "asymptotic prediction": ct_inf.round(2)}))
print(f"mean |observed - finite-sample| = {np.mean(np.abs(ct_obs - ct_n)):.2f}   vs   mean |observed - asymptotic| = {np.mean(np.abs(ct_obs - ct_inf)):.2f}")

# ---- clinical usefulness ----
ARMS_C = ["share_score_ct", "IW_donor", "C_target", "local_distill"]
C = R[R.arm.isin(ARMS_C)].groupby(CELL + ["arm"])[["brier_y", "auc", "cal_slope", "citl", "nb10", "nb15", "nb20"]].mean().round(4)
print("\nCalibration and net benefit (thresholds 10/15/20%) on the real test patients"); display(C)